<a href="https://colab.research.google.com/github/Doris-QZ/Fine-Tuning-BERT-Large-with-QLoRA-for-Author-Identification/blob/main/1_QLoRA_r8a4_AllLin_Author_Identification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Introduction

In this notebook, we fine-tune the **BERT Large** model using **QLoRA** on the  [**Spooky Author Identification**](https://www.kaggle.com/competitions/spooky-author-identification) dataset from Kaggle. The dataset contains excerpts from horror stories written by three authors: **Edgar Allan Poe**, **HP Lovecraft** and **Mary Shelley**, with a total of **19,579 samples** in the training set.  

In this project, we experiment with different **hyperparameter settings**:
> r ∈ {8, 16}  
> a ∈ {2, 4, 8, 12}   
> lora_dropout ∈ {0.1, 0.2}  
> learning_rate ∈ {1e-4, 2e-4, 2.5e-4}  
> lr_scheduler_type ∈ {'linear', 'cosine'}   
> weight_decay ∈ {0.0, 0.01}  
> target_modules ∈ {Q+K+V, all attention, all linear}

After evaluating all models, we keep the **best two** for the final **ensemble learning**. This notebook presents the **best single model**, with the following hyperparameter configuration:
> r=8  
> a=4  
> lora_dropout=0.2  
> learning_rate=2e-4  
> lr_scheduler_type='linear'(default)  
> weight_decay=0.01  
> target_modules= all linear (['query', 'key', 'value', 'output.dense', 'intermediate.dense'])

<br>

**Data Source**: Meg Risdal and Rachael Tatman. Spooky Author Identification. https://kaggle.com/competitions/spooky-author-identification, 2017. Kaggle.

**Connect to Kaggle and download the dataset**

In [ ]:
# Install Kaggle library
!pip install kaggle

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Make a directory for Kaggle
!mkdir ~/.kaggle

# Copy the kaggle.json file to the directory
!cp /content/drive/MyDrive/ColabNotebooks/Kaggle_API_Key/kaggle.json ~/.kaggle/

# Change the file permission to read/write to the owner only
!chmod 600 ~/.kaggle/kaggle.json

# Download the dataset
!kaggle competitions download spooky-author-identification

# Unzip the data
!unzip spooky-author-identification.zip
!unzip train.zip
!unzip test.zip

In [ ]:
!pip install -q -U bitsandbytes transformers accelerate peft

In [3]:
import bitsandbytes
import transformers
import peft
import accelerate

print(f"bitsandbytes: {bitsandbytes.__version__}")
print(f"transformers: {transformers.__version__}")
print(f"peft: {peft.__version__}")
print(f"accelerate: {accelerate.__version__}")

bitsandbytes: 0.48.1
transformers: 4.57.0
peft: 0.17.1
accelerate: 1.10.1


In [4]:
# Import packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch.optim import AdamW
from transformers import AutoTokenizer, BertForSequenceClassification, BitsAndBytesConfig
from transformers import Trainer, TrainingArguments, DataCollatorWithPadding
from transformers import EarlyStoppingCallback
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from peft import TaskType, replace_lora_weights_loftq
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

### Dataset Preparation

Since I already performed **Exploratory Data Analysis** in another [notebook](https://github.com/Doris-QZ/spooky_author_identification/blob/main/0_Hybrid_Spooky_Author_Identification.ipynb), I’ll skip the EDA section here and go straight to preparing the dataset for the model.

In [5]:
# Load the data
train = pd.read_csv('/content/train.csv')
test = pd.read_csv('/content/test.csv')

# Take a look at the data
train.info()
train.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19579 entries, 0 to 19578
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   id      19579 non-null  object
 1   text    19579 non-null  object
 2   author  19579 non-null  object
dtypes: object(3)
memory usage: 459.0+ KB


,id,text,author
0,id26305,"This process, however, afforded me no means of...",EAP
1,id17569,It never once occurred to me that the fumbling...,HPL
2,id11008,"In his left hand was a gold snuff box, from wh...",EAP
3,id27763,How lovely is spring As we looked from Windsor...,MWS
4,id12958,"Finding nothing else, not even gold, the Super...",HPL


In [6]:
# Create a new column of 'label' to encode 'author' to numeric
label_to_id = {'EAP': 0, 'MWS': 1, 'HPL': 2}
train['labels'] = train['author'].map(label_to_id)

In [7]:
# Split the 'train' data into training and validation dataset
train_set, val_set = train_test_split(train, test_size=0.1, random_state=42)

In [ ]:
# Load the pretrained tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-large-uncased")

In [9]:
# Tokenize text data
train_tokenized = tokenizer(train_set['text'].tolist(),
                            max_length=512,
                            truncation=True,
                            padding=False)

val_tokenized = tokenizer(val_set['text'].tolist(),
                          max_length=512,
                          truncation=True,
                          padding=False)

In [11]:
class TextDataset(torch.utils.data.Dataset):
    """
    Create a PyTorch Dataset object from tokenized text data and labels.

    Args:
        data (dict): Dictionary of lists from deberta-v3-large tokenizer(without padding)
        label (list): None or List of numerical labels

    Returns:
        dict[str, torch.tensor]: A dictionary containing a single sample's tensors.

    """
    def __init__(self, data, label=None):
        self.data = data
        self.label = label

    def __len__(self):
        return len(self.data['input_ids'])

    def __getitem__(self, idx):
        item = {'input_ids': torch.tensor(self.data['input_ids'][idx]),
                'token_type_ids': torch.tensor(self.data['token_type_ids'][idx]),
                'attention_mask': torch.tensor(self.data['attention_mask'][idx])}
        if self.label:
            item['labels'] = torch.tensor(self.label[idx])
        return item

In [12]:
train_dataset = TextDataset(train_tokenized, train_set['labels'].tolist())
val_dataset = TextDataset(val_tokenized, val_set['labels'].tolist())
train_dataset[0]

{'input_ids': tensor([  101,  2145,  2500,  1010,  2164,  3533,  2370,  1010,  2031,  8106,
          2205,  3748,  1998, 10392,  2005, 17358, 13675, 14728,  5897,  1012,
           102]),
 'token_type_ids': tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]),
 'attention_mask': tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]),
 'labels': tensor(2)}

### Model Preparation

To fine-tune a large language model using QLoRA within the Hugging Face ecosystem, the process typically involves three main steps:
1. Set the quantization configuration -> Load the quantized model
2. Prepare the model for k-bit training
3. Set the LoRA configuration -> Load the PEFT model with the LoRA configuration

In [ ]:
# Set quantization configuration
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

# Load the quantized model
model = BertForSequenceClassification.from_pretrained(
    "bert-large-uncased",
    num_labels=3,
    quantization_config=quantization_config,
    device_map='auto'
)

In [15]:
# Prepare the model for k-bit training
model = prepare_model_for_kbit_training(model)
model

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 1024, padding_idx=0)
      (position_embeddings): Embedding(512, 1024)
      (token_type_embeddings): Embedding(2, 1024)
      (LayerNorm): LayerNorm((1024,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-23): 24 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear4bit(in_features=1024, out_features=1024, bias=True)
              (key): Linear4bit(in_features=1024, out_features=1024, bias=True)
              (value): Linear4bit(in_features=1024, out_features=1024, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear4bit(in_features=1024, out_features=1024, bias=True)
              (LayerNor

In [16]:
# Set Lora configuration
lora_config = LoraConfig(
    r=8,
    lora_alpha=4,
    lora_dropout=0.2,
    target_modules=['query', 'key', 'value', 'output.dense', 'intermediate.dense'],
    bias='none',
    task_type=TaskType.SEQ_CLS
)

# Load the PEFT model with the LoRA configuration
r8a4_AllLin = get_peft_model(model, lora_config)

In [17]:
# Reinitialize LoRA weights using LoftQ
replace_lora_weights_loftq(r8a4_AllLin)

In [18]:
# Unfreeze the classifier weights
for name, param in r8a4_AllLin.named_parameters():
    if 'classifier' in name:
        param.requires_grad = True

In [19]:
r8a4_AllLin.print_trainable_parameters()

trainable params: 3,545,094 || all params: 338,686,982 || trainable%: 1.0467


In [ ]:
# Create data collator
data_collator = DataCollatorWithPadding(tokenizer=tokenizer, pad_to_multiple_of=8)

# Define the metrics
def compute_metrics(eval_pred):
    y_pred, y_true = eval_pred
    y_pred = np.argmax(y_pred, axis=1)
    accuracy = accuracy_score(y_true, y_pred)
    return {'accuracy': accuracy}

In [ ]:
# Set up the training arguments
training_args = TrainingArguments(
    output_dir='/content/drive/MyDrive/ColabNotebooks/Spooky_Author_Identification/qlora/r8a4_AllLin',
    num_train_epochs=20,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    weight_decay=0.01,
    max_grad_norm=0.3,
    eval_strategy='epoch',
    logging_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    gradient_checkpointing=True,
    report_to = "none"
)

In [ ]:
# Set up the trainer
trainer = Trainer(
    model=r8a4_AllLin,
    args=training_args,
    data_collator=data_collator,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

r8a4_AllLin.config.use_cache = False

In [ ]:
# Training
training_log = trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.732200,0.578797,0.760981
2,0.446400,0.425778,0.825332
3,0.359500,0.398774,0.836057
4,0.287000,0.438987,0.832993
5,0.220400,0.355973,0.865169
6,0.165400,0.385648,0.865679
7,0.125000,0.480504,0.862615
8,0.098700,0.458517,0.869765
